In [ ]:
import os

import json
import math
import pickle
import networkx as nx
import pandas as pd
import numpy as np

import seaborn as sns
import networkx as nx
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from data.reference.residue_dictionary import residue3to1 as AA3TO1
from data.reference.residue_dictionary import residue1to3 as AA1TO3

import matplotlib.pyplot as plt
from utils.functions import load_yaml
from utils.graphfunction import load_graph, get_sample, get_uniprot_from_nodes, get_pos_from_nodes, get_res_from_nodes, get_node_id_rm_copy, get_only_copy_node

from vizutils import *

from pyaaindex.api import get_features, to_frame, get_aa_delta
from helper import get_aainfo, add_copy_feat

In [ ]:
config = load_yaml("../config/RRGCL.yaml")
DATABASE = config.DATABASE

aa_seq = np.array(['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y'])

In [ ]:
# Graph Load
# finalG = load_graph((f"{DATABASE}/skipCDSFilter_weighted_all-chain_v031626.pkl"))
finalG = load_graph((f"{DATABASE}/cleaned_weighted_graph_041226.pkl"))
node_in_G = list(finalG.nodes())
node_df_inG = pd.DataFrame(node_in_G, columns=['node_id'])
print(finalG)

# origin Data
mut_df = pd.read_csv((f"{DATABASE}/node_mutation_occurrence_essential.csv"))
y_df = pd.read_csv(f"{DATABASE}/node_features_with_location_nodeid_v031026.csv")
edge_df = nx.to_pandas_edgelist(finalG)

# Processed Data
# ss_df = pd.read_csv(f"{DATABASE}/processed_ss_df_v040126.csv")
ss_df = pd.read_csv(f"{DATABASE}/processed_ss_df_v041026.csv")
gnomad_mut_df = pd.read_csv((f"{DATABASE}/gnomad_mutation_counts_freq_final.csv"))

# AlphaMissense
with open(f"{DATABASE}/am_features.json", 'rb') as f:
    am = json.load(f)

In [ ]:
copy_nodes = get_only_copy_node(node_in_G)

# Filtering Non-reviewed Protein

In [ ]:
# Secondary Structure Missing Value
with open('reference/ss_nan_nodes.json', 'r') as f:
    ss_nan_node_in_G = json.load(f)

with open('reference/pssm_nan_nodes.json', 'r') as f:
    pssm_nan_node_in_G = json.load(f)

with open('reference/hmm_nan_nodes.json', 'r') as f:
    hmm_nan_node_in_G = json.load(f)

with open('reference/am_nan_nodes.json', 'r') as f:
    am_nan_node_in_G = json.load(f)

merged_nan_nodes = set(ss_nan_node_in_G).union(pssm_nan_node_in_G).union(hmm_nan_node_in_G).union(am_nan_node_in_G)
print(len(merged_nan_nodes))

In [ ]:
rm_uniprot = [
    "a0a1x8xl64", # AM
    "b4dr52", # AM
    "b2r5b3", # AM
]

In [ ]:
proc_edge_df = pd.read_csv(f"{DATABASE}/step5_rmStrangeEdge_exceptCDSFilter_human_aa_edges_exclubq_Nucleosome_related_data_v031626.csv")
proc_edge_df.head(3)
len(proc_edge_df)

In [ ]:
proc_edge_df.head(3)

In [ ]:
rm_nonReview_df = proc_edge_df[
    (~proc_edge_df['remove_homo_uniprot1'].isin(rm_uniprot)) & 
    (~proc_edge_df['remove_homo_uniprot2'].isin(rm_uniprot))
]

rm_nonReview_df = rm_nonReview_df[
    (~rm_nonReview_df['nodeid_1'].isin(merged_nan_nodes)) &
    (~rm_nonReview_df['nodeid_2'].isin(merged_nan_nodes))
]

len(rm_nonReview_df)

In [ ]:
from generation.graph.helper import GenerateGraph_from_edge
newG = GenerateGraph_from_edge(rm_nonReview_df, src='nodeid_1', tar='nodeid_2', weight_col='cleaned_total_energy')
print(newG)

In [ ]:
node_in_G = list(newG.nodes())
node_df_inG = pd.DataFrame(node_in_G, columns=['node_id'])

In [ ]:
with open(os.path.join(DATABASE, 'cleaned_weighted_graph_041226.pkl'), 'wb') as f:
    pickle.dump(newG, f)

In [ ]:
finalG = load_graph(os.path.join(DATABASE, 'cleaned_weighted_graph_041226.pkl'))

In [ ]:
print(finalG)

# Graph EDA

In [ ]:
print(finalG)
cc_list = list(nx.connected_components(finalG))
len(cc_list)

In [ ]:
n_in_cc = []
for cc_ in cc_list:
    n_in_cc.append(len(cc_))
n_in_cc.sort(reverse=True)

print("Top3 Size", n_in_cc[:3])

In [ ]:
plt.figure(figsize=(7,3))
plt.hist(n_in_cc[3:], color='skyblue', edgecolor='black', bins=100)
plt.yscale('log')
plt.show()

# Mutation Profile

## AAindex

### AAindex1

In [ ]:
aaidx1_list = [
    'KYTJ820101',  # Hydropathy index (Kyte-Doolittle)
    'KLEP840101',  # Net charge (Klein et al.)
    'BHAR880101',  # Average flexibility indices (Bhaskaran-Ponnuswamy)
    'JANJ780101',  # Free energy of transfer (Janin)
    'CHOP780201',  # Conformational parameter beta-turn (Chou-Fasman)
    'GRAR740102',  # Polarity (Grantham)
    'GRAR740103'   # Volume (Grantham)
]

In [ ]:
aaidx1 = get_features(aaidx1_list)['idx1']

In [ ]:
aaidx1.head(3)

In [ ]:
corr_matrix = aaidx1.corr()
plt.figure(figsize=(10, 8))

sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', linewidths=0.5)

plt.title('Correlation Heatmap of AAindex1 Properties', fontsize=15)
plt.tight_layout()

# plt.savefig('aaindex1_heatmap.png')
plt.show()

In [ ]:
delta_df = get_aa_delta("KYTJ820101")
delta_df

### AAindex2

In [ ]:
ids = [
    'CSEM940101', # Replaceability
    'GEOD900101', # Hydrophobicity Scoring Matrix
    'GRAR740104', # Chemical Distance
    'JOHM930101', # Strcture-based amino acid scoring Table
    'MIYS930101', # Base-substitution-protein-stability matrix
    'QU_C930103', # Mutant distance based on spatial preference factor
    'GONG920101', # Mutation Matrix for initailly aligning
    # 'KLEP840101' # Net Chard --> First Index (AAindex1)
]
aaidx2 = get_features(ids)

In [ ]:
frames = to_frame(aaidx2['idx2'])
frame_list = []
for feature_id, df in frames.items():
    frame_list.append(df)

In [ ]:
feature_vectors = {}
for matrix_id in ids:
    feature_vectors[matrix_id] = get_upper_triangle_values(frames[matrix_id])

df_features = pd.DataFrame(feature_vectors)
corr_result = df_features.corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr_result, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Similarity between Mutation Profiles (Feature-wise)")
plt.show()

## Alpha Missense

In [ ]:
list(am.keys())[:10]

In [ ]:
# Homodimer feature Copy
updated_am, fail_am = add_copy_feat(am, copy_nodes)
am_nan_node_in_G = list(set(node_in_G) - set(updated_am))

print(len(am_nan_node_in_G))

In [ ]:
# with open('reference/am_nan_nodes.json', 'w') as f:
#     json.dump(am_nan_node_in_G, f)

In [ ]:
f = np.array([get_uniprot_from_nodes(f) for f in am_nan_node_in_G])
print(len(set(f)))
f

In [ ]:
sample = list(am.keys())[::10000][:10]
sample

In [ ]:
scaler = StandardScaler()

feat_for_sim_1 = []
feat_for_sim_2 = []

# AlphaMissense
mask = (aa_seq != res_1) & (aa_seq != res_2)

am_1 = np.array(am[get_node_id_rm_copy(target_1)])[mask]
am_2 = np.array(am[get_node_id_rm_copy(target_2)])[mask]

# Merge & Normalization 
feat_for_sim_1.append(am_1)
feat_for_sim_2.append(am_2)

feat_for_sim_1 = (feat_for_sim_1 - np.mean(feat_for_sim_1, axis=1, keepdims=True)) / (np.std(feat_for_sim_1, axis=1, keepdims=True) + 1e-8)
feat_for_sim_2 = (feat_for_sim_2 - np.mean(feat_for_sim_2, axis=1, keepdims=True)) / (np.std(feat_for_sim_2, axis=1, keepdims=True) + 1e-8)

## Gnomad Mutation

### Manual ADD Mutation Information

In [ ]:
# from generation.mutation.gnomad.helper import process_gnomad_csv
# add_files = ['O60814_H2BC12_gnomAD_v4.1.1_ENST00000356950.csv',
#              'P11388_TOP2A_gnomAD_v4.1.1_ENST00000423485.csv',
#              'Q5EE01_CENPW_gnomAD_v4.1.1_ENST00000368328.csv',
#              'Q96KQ7_EHMT2_gnomAD_v4.1.1_ENST00000375537.csv'
#              ]

In [ ]:
# c_gnomad_mut_df = gnomad_mut_df.copy()
# for file in add_files:
#     print(file)
#     uniprot = file.split("_")[0]
#     ENST = file.split("_")[-1].split(".")[0]
#     gene_symbol= file.split("_")[1]
#     df_raw = pd.read_csv(f"reference/{file}")
#     proc_df = process_gnomad_csv(df_raw, uniprot, ENST, gene_symbol=gene_symbol)
#     c_gnomad_mut_df = pd.concat([c_gnomad_mut_df, proc_df], axis=0)
# c_gnomad_mut_df.to_csv('proc_data/gnomad_mutation_counts_freq_final.csv', index=False)
# c_gnomad_mut_df.tail(5)

### EDA

In [ ]:
gnomad_mut_df.head(4)

In [ ]:
covered_nodes = []
node_in_gnomad = gnomad_mut_df['node_id'].tolist()

for n in node_in_G:
    cleaned_node_id = get_node_id_rm_copy(n)
    if cleaned_node_id in node_in_gnomad:
        covered_nodes.append(n)

In [ ]:
print(len(covered_nodes), "||", f"{round((len(covered_nodes) / len(node_in_G))*100, 2)}%")

In [ ]:
len(gnomad_mut_df.uniprot_id.unique())

# Secondary Strcture

In [ ]:
float_cols = ['node_id', 
              'rel_sasa', 'ss_helix', 'ss_sheet','ss_loop', 'depth', 'hse_up', 'hse_down',
              'dssp_accessibility', 'dssp_TCO', 'dssp_kappa','dssp_alpha', 'dssp_phi', 'dssp_psi',]

str_cols = ['node_id', 
            'dssp_sec_struct', # Class
            'dssp_helix_3_10', 'dssp_helix_alpha', 'dssp_helix_pi', 'dssp_helix_pp', 'dssp_bend',
            'dssp_chirality', 'dssp_sheet', 'dssp_strand', 'dssp_ladder_1','dssp_ladder_2',]

all_cols = list(set(float_cols).union(str_cols))
all_cols.remove('node_id')

In [ ]:
ss_apply_G = pd.merge(node_df_inG, ss_df, on='node_id', how='left')
ss_apply_G

In [ ]:
for col in float_cols:
    print(col, round((len(ss_apply_G[~ss_apply_G[col].isna()])/len(ss_apply_G))*100,2),'%')

print("-----------------------------")
for col in str_cols:
    print(col, round((len(ss_apply_G[~ss_apply_G[col].isna()])/len(ss_apply_G))*100,2),'%')

In [ ]:
ss_apply_G['dssp_helix_pi'].value_counts()

In [ ]:
ss_apply_G['dssp_helix_pi'].value_counts()

In [ ]:
ss_nan_nodes = ss_apply_G[ss_apply_G[all_cols].isna().all(axis=1)].node_id.tolist()
nan_uniprot = set([get_uniprot_from_nodes(n) for n in ss_nan_nodes])
print("NaN Node:", len(ss_nan_nodes), nan_uniprot)

In [ ]:
# with open('reference/ss_nan_nodes.json', 'w') as f:
#     json.dump(ss_nan_nodes, f)

## float EDA

In [ ]:
float_ss_df = ss_apply_G[float_cols]
float_ss_df.info()

In [ ]:
float_ss_df_inG = float_ss_df[float_ss_df['node_id'].isin(node_in_G)]
print(len(node_in_G))
float_ss_df_inG.info()

In [ ]:
viz_cols = list(set(float_cols).difference(['node_id']))
plot_distribution_subplots(float_ss_df_inG, viz_cols, cols_per_row=3, log_scale=True)

## str EDA

In [ ]:
str_ss_df = ss_apply_G[str_cols]
str_ss_df.info()

In [ ]:
str_viz_cols = list(set(str_cols).difference(['node_id']))

plot_value_counts(str_ss_df, str_viz_cols, cols_per_row=2, top_n=15)

# Evolutionary Information

In [ ]:
with open('proc_data/pssm_features.json') as f:
    #  Information Content(Entropy, highscore-preservative) 1 + A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y (log-odd score 20)==> len 21
    pssm = json.load(f)

with open('proc_data/hmm_features.json') as f:
    # A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y
    hmm = json.load(f)

## PSSM EDA

In [ ]:
final_pssm, failed_pssm_node = add_copy_feat(pssm, copy_nodes)
print("Failed_node", len(failed_pssm_node))
node_in_pssm = list(final_pssm.keys())

In [ ]:
node_not_in_pssm = set(node_in_G).difference(node_in_pssm)
print(len(node_not_in_pssm))

In [ ]:
node_not_in_pssm

In [ ]:
# with open('reference/pssm_nan_nodes.json', 'w') as f:
#     json.dump(node_not_in_pssm, f, default=list)

## HMM EDA

In [ ]:
updated_hmm, fail = add_copy_feat(hmm, copy_nodes)
print("Failed Node", len(fail))
node_in_hmm = list(updated_hmm.keys())
print(len(set(node_in_G).difference(node_in_hmm)))

In [ ]:
node_not_in_hmm = set(node_in_G).difference(updated_hmm)
print(len(node_not_in_hmm))

In [ ]:
node_not_in_hmm

In [ ]:
# with open('reference/hmm_nan_nodes.json', 'w') as f:
#     json.dump(node_not_in_hmm, f, default=list)

# Merge Dataset

In [ ]:
node_df_inG['uniprot'] = node_df_inG['node_id'].apply(lambda x: x.split('_')[0].split('-')[0])
node_df_inG['copy_idx'] = (
    node_df_inG['node_id']
    .str.split('_').str[0]
    .str.split('-').str[1]
    .fillna(0)
)
node_df_inG['pos'] = node_df_inG['node_id'].apply(lambda x: x.split('_')[1])
node_df_inG['res'] = node_df_inG['node_id'].apply(lambda x: AA3TO1[x.split('_')[2].upper()].upper())

In [ ]:
node_df_inG

In [ ]:
cp_incl_am, fail_am = add_copy_feat(am, copy_nodes)
amino_acids = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 
               'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
am_columns = [f'am_{aa}' for aa in amino_acids]
am_df = pd.DataFrame.from_dict(cp_incl_am, orient='index', columns=am_columns)
am_df = am_df.reset_index().rename(columns={'index': 'node_id'})
am_df.head(2)

In [ ]:
mutation_df = mut_df[['node_id', 'unique_mutation_types_count', 'total_mutations_count', 'unique_patients_count']]
mutation_df.rename({'unique_mutation_types_count': 'c_unique_mutation_types_count',
                    'total_mutations_count': 'c_total_mutations_count',
                    'unique_patients_count': 'c_unique_patients_count'}, axis=1, inplace=True)

mutation_df.head(3)

In [ ]:
gnomad = gnomad_mut_df[['node_id', 'unique_mutation_type', 'total_mutations_count', 'frequency']]
gnomad.rename({'unique_mutation_type': 'gm_unique_mutation_types_count',
               'total_mutations_count': 'gm_total_mutations_count',
               'frequency': 'gm_mutation_frequency'}, axis=1, inplace=True)
gnomad.head(2)

In [ ]:
aaidx1_list = [
    'KYTJ820101',  # Hydropathy index (Kyte-Doolittle)
    'KLEP840101',  # Net charge (Klein et al.)
    'BHAR880101',  # Average flexibility indices (Bhaskaran-Ponnuswamy)
    'JANJ780101',  # Free energy of transfer (Janin)
    'CHOP780201',  # Conformational parameter beta-turn (Chou-Fasman)
    'GRAR740102',  # Polarity (Grantham)
    'GRAR740103'   # Volume (Grantham)
]
aaidx1 = get_features(aaidx1_list)['idx1']
aaidx1 = aaidx1.rename(columns={col: f'aa1_{col}' for col in aaidx1.columns if col != 'node_id'})
aaidx1 = aaidx1.reset_index()
aaidx1.rename({'index':'aa1'}, axis=1, inplace=True)

In [ ]:
merged_df = pd.merge(node_df_inG, ss_df, on='node_id', how='left')
merged_df = pd.merge(merged_df, gnomad, on='node_id', how='left')
merged_df = pd.merge(merged_df, am_df, on='node_id', how='left')

merged_df['aa1'] = merged_df['node_id'].apply(lambda x: AA3TO1[x.split('_')[2].upper()].upper())
merged_df = pd.merge(merged_df, aaidx1, on='aa1', how='left')
merged_df.head(3)

In [ ]:
ids2 = [
    'CSEM940101', # Replaceability
    'GEOD900101', # Hydrophobicity Scoring Matrix
    'GRAR740104', # Chemical Distance
    'JOHM930101', # Strcture-based amino acid scoring Table
    'MIYS930101', # Base-substitution-protein-stability matrix
    'QU_C930103', # Mutant distance based on spatial preference factor
    'GONG920101', # Mutation Matrix for initailly aligning
    # 'KLEP840101' # Net Chard --> First Index (AAindex1)
]
aaidx2 = get_features(ids2)
frames = to_frame(aaidx2['idx2'])
frame_list = []
for feature_id, df in frames.items():
    df = df.reset_index()
    df.rename({'index':'aa1'}, axis=1, inplace=True)
    frame_list.append(df)

In [ ]:
for aa2_df, feat_name in zip(frame_list, ids2):
    aa2_df.rename(columns={aa: f'aa2_{feat_name}_{aa}' for aa in aa2_df.columns if aa != 'aa1'}, inplace=True)
    merged_df = pd.merge(merged_df, aa2_df, on='aa1', how='left')

In [ ]:
merged_df

In [ ]:
merged_df.info()

In [ ]:
pssm_columns = [f'pssm_{aa}' for aa in amino_acids]
pssm_columns.append('pssm_entropy')
pssm_df = pd.DataFrame.from_dict(final_pssm, orient='index', columns=pssm_columns)
pssm_df = pssm_df.reset_index().rename(columns={'index': 'node_id'})
pssm_df.head(2)

In [ ]:
hmm_columns = [f'hmm_{aa}' for aa in amino_acids]
hmm_columns.extend(['hmm_MM', 'hmm_MI', 'hmm_MD', 'hmm_IM', 'hmm_II', 'hmm_DM', 'hmm_DD', 'hmm_neff'])
hmm_df = pd.DataFrame.from_dict(updated_hmm, orient='index', columns=hmm_columns)
hmm_df = hmm_df.reset_index().rename(columns={'index': 'node_id'})
hmm_df.head(2)

In [ ]:
merged_df = pd.merge(merged_df, pssm_df, on='node_id', how='left')
merged_df = pd.merge(merged_df, hmm_df, on='node_id', how='left')

In [ ]:
merged_df

In [ ]:
merged_df.info()

In [ ]:
merged_df.to_csv(f"{DATABASE}/merged_feature_data_v041226.csv", index=False)